# Lab 11: APIM + Managed Identity로 Microsoft Graph 호출

이 노트북은 클라이언트가 **APIM 구독 키만으로** Microsoft Graph를 조회할 수 있음을 검증합니다.
클라이언트는 Graph용 Bearer 토큰을 **직접 발급하지 않습니다** — APIM의 Managed Identity가
내부에서 토큰을 발급해 백엔드로 주입합니다.

- **Part 1 (옵션 A)**: System MI 1개 + 정책 게이트 → 구독별 Operation을 403으로 차단
- **Part 2 (옵션 B)**: 권한별 UAMI 3개 → 잘못된 경로는 토큰 자체에 권한이 없어 Graph가 거부

> 실행 전 `.env`에 `APIM_KEY_GRAPH_USERS`, `APIM_KEY_GRAPH_MAIL`, `APIM_KEY_GRAPH_SHAREPOINT`가 입력되어 있어야 합니다.

In [17]:
# 셀 1: 환경 변수 로드
import os, requests
from pathlib import Path

def load_env(path=".env"):
    env = {}
    p = Path(path)
    if not p.exists():
        p = Path("../../.env")
    if p.exists():
        for line in p.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env()
APIM_URL = env.get("APIM_URL", "").rstrip("/")
KEY_USERS = env.get("APIM_KEY_GRAPH_USERS", "")
KEY_MAIL = env.get("APIM_KEY_GRAPH_MAIL", "")
KEY_SHAREPOINT = env.get("APIM_KEY_GRAPH_SHAREPOINT", "")
SP_HOSTNAME = env.get("SHAREPOINT_SITE_HOSTNAME", "")
SP_PATH = env.get("SHAREPOINT_SITE_PATH", "")
TEST_USER_ID = env.get("GRAPH_TEST_USER_ID", "")  # 비우면 셀 3에서 첫 사용자 id 자동 사용

assert APIM_URL, "APIM_URL이 .env에 없습니다."
assert KEY_USERS and "<" not in KEY_USERS, "APIM_KEY_GRAPH_USERS를 .env에 입력하세요."
assert KEY_MAIL and "<" not in KEY_MAIL, "APIM_KEY_GRAPH_MAIL을 .env에 입력하세요."
print("APIM_URL:", APIM_URL)
print("graph-users 키 로드:", KEY_USERS[:4] + "..." )
print("graph-mail  키 로드:", KEY_MAIL[:4] + "...")
print("graph-sharepoint 키 로드:", (KEY_SHAREPOINT[:4] + "...") if KEY_SHAREPOINT else "(미설정)")

GRAPH_BASE = f"{APIM_URL}/graph"
print("Graph API base:", GRAPH_BASE)

APIM_URL: https://apim-ai-gw-aigateway-20260716.azure-api.net
graph-users 키 로드: 7c5d...
graph-mail  키 로드: 0e88...
graph-sharepoint 키 로드: b2bc...
Graph API base: https://apim-ai-gw-aigateway-20260716.azure-api.net/graph


## 개념 확인: 클라이언트는 Graph 토큰을 보내지 않는다

아래 모든 요청은 `Ocp-Apim-Subscription-Key` 헤더만 사용합니다.
`Authorization: Bearer ...` 헤더는 **어디에도 없습니다**. Graph 토큰 발급은 APIM MI의 몫입니다.

In [18]:
# 셀 2: 요청 헬퍼 — 구독 키만 사용 (Bearer 토큰 없음)
def graph_get(path, sub_key, params=None):
    url = f"{GRAPH_BASE}{path}"
    headers = {"Ocp-Apim-Subscription-Key": sub_key}  # Graph Bearer 토큰 없음!
    r = requests.get(url, headers=headers, params=params, timeout=30)
    print(f"GET {path}  → HTTP {r.status_code}")
    assert "authorization" not in {k.lower() for k in headers}, "클라이언트가 Bearer를 보내면 안 됩니다"
    return r

print("헬퍼 준비 완료. 클라이언트 요청 헤더에는 Graph Bearer 토큰이 없습니다.")

헬퍼 준비 완료. 클라이언트 요청 헤더에는 Graph Bearer 토큰이 없습니다.


## Part 1 (옵션 A): System MI + 정책 게이트

`graph-users` 구독은 `/users`만, `graph-mail` 구독은 `/users/{id}/messages`만 허용됩니다.
격리는 **APIM 정책**이 담당합니다 (실제 토큰 권한은 System MI에 모두 존재).

In [19]:
# 셀 3: [graph-users 키] GET /users → 200 기대
r = graph_get("/users", KEY_USERS, params={
    "$top": 5,
    "$select": "id,displayName,userPrincipalName,mail",
})
assert r.status_code == 200, f"기대 200, 실제 {r.status_code}: {r.text[:300]}"
users = r.json().get("value", [])

print(f"✅ 사용자 {len(users)}명 조회 성공 — 클라이언트는 구독 키만 전송 (Graph 토큰 없음)\n")
print(f"{'#':<3}{'displayName':<24}{'userPrincipalName':<40}{'mail'}")
print("-" * 96)
for i, u in enumerate(users, 1):
    print(f"{i:<3}{(u.get('displayName') or '-'):<24}{(u.get('userPrincipalName') or '-'):<40}{u.get('mail') or '-'}")

print("\nid 목록:")
for u in users:
    print(" -", u.get("id"))


TEST_USER_ID = users[0].get("id")
print("\n다음 테스트에 사용할 TEST_USER_ID:", TEST_USER_ID)

GET /users  → HTTP 200
✅ 사용자 5명 조회 성공 — 클라이언트는 구독 키만 전송 (Graph 토큰 없음)

#  displayName             userPrincipalName                       mail
------------------------------------------------------------------------------------------------
1  System Administrator    admin@MngEnvMCAP094463.onmicrosoft.com  admin@MngEnvMCAP094463.onmicrosoft.com
2  Changju Ahn             changjuahn@MngEnvMCAP094463.onmicrosoft.comchangjuahn@MngEnvMCAP094463.onmicrosoft.com
3  github01                github01@MngEnvMCAP094463.onmicrosoft.com-
4  github02                github02@MngEnvMCAP094463.onmicrosoft.com-
5  github03                github03@MngEnvMCAP094463.onmicrosoft.com-

id 목록:
 - cb319021-e447-4dfb-837a-60042cc7a8a9
 - 2cb99f43-e40c-47b9-8891-2493827c716c
 - 3c3e70a8-e00e-4573-8f05-e67689305650
 - 37612f1f-23ed-4114-ae68-98c65b416d5b
 - 8c4390e0-f89b-451f-94a9-776129710174

다음 테스트에 사용할 TEST_USER_ID: cb319021-e447-4dfb-837a-60042cc7a8a9


In [20]:
# 셀 4: [graph-users 키] GET /users/{id}/messages → 403 기대 (정책 차단)
r = graph_get(f"/users/{TEST_USER_ID}/messages", KEY_USERS)
assert r.status_code == 403, f"기대 403(정책 차단), 실제 {r.status_code}: {r.text[:300]}"
print("✅ 예상대로 차단됨 — graph-users 구독은 메일 조회 불가 (APIM 정책)")

GET /users/cb319021-e447-4dfb-837a-60042cc7a8a9/messages  → HTTP 403
✅ 예상대로 차단됨 — graph-users 구독은 메일 조회 불가 (APIM 정책)


In [21]:
# 셀 5: [graph-mail 키] 최근 메일 5개 제목 조회 → 200 기대
r = graph_get(
    f"/users/{TEST_USER_ID}/messages",
    KEY_MAIL,
    params={
        "$top": 5,
        "$select": "subject,receivedDateTime",
        "$orderby": "receivedDateTime desc",
    },
)
assert r.status_code == 200, f"기대 200, 실제 {r.status_code}: {r.text[:300]}"
msgs = r.json().get("value", [])
print(f"메시지 {len(msgs)}건 조회 성공")

for i, msg in enumerate(msgs, 1):
    print(f"{i}. [{msg.get('receivedDateTime')}] {msg.get('subject') or '(제목 없음)'}")

GET /users/cb319021-e447-4dfb-837a-60042cc7a8a9/messages  → HTTP 200
메시지 5건 조회 성공
1. [2026-07-28T05:58:42Z] PIM: System Administrator activated the Global Administrator role assignment
2. [2026-07-28T05:58:39Z] PIM: Your Global Administrator role is now active in Microsoft Entra ID
3. [2026-07-27T06:10:57Z] PIM: System Administrator activated the Global Administrator role assignment
4. [2026-07-27T06:10:57Z] PIM: Your Global Administrator role is now active in Microsoft Entra ID
5. [2026-07-27T01:12:59Z] See what happened in the 2 managed environments this week.


In [14]:
# 셀 6: [graph-mail 키] GET /users → 403 기대 (정책 차단)
r = graph_get("/users", KEY_MAIL)
assert r.status_code == 403, f"기대 403(정책 차단), 실제 {r.status_code}: {r.text[:300]}"
print("✅ 예상대로 차단됨 — graph-mail 구독은 사용자 목록 조회 불가 (APIM 정책)")

GET /users  → HTTP 403
✅ 예상대로 차단됨 — graph-mail 구독은 사용자 목록 조회 불가 (APIM 정책)


## Part 1 계속: SharePoint 리스트 조회 (graph-sharepoint 키)

`graph-sharepoint` 구독은 `Sites.Read.All` 권한을 통해 SharePoint 사이트의 리스트/항목을 조회합니다.
site-id는 노트북이 hostname+path로 런타임에 해석합니다.

In [22]:
# 셀 7: [graph-sharepoint 키] 사이트 해석 → GET /sites/{host}:/sites/{path}
assert KEY_SHAREPOINT and "<" not in KEY_SHAREPOINT, "APIM_KEY_GRAPH_SHAREPOINT를 .env에 입력하세요."
assert SP_HOSTNAME and "<" not in SP_HOSTNAME, "SHAREPOINT_SITE_HOSTNAME을 .env에 입력하세요."
assert SP_PATH and "<" not in SP_PATH, "SHAREPOINT_SITE_PATH를 .env에 입력하세요."

site_path = f"/sites/{SP_HOSTNAME}:/sites/{SP_PATH}"
r = graph_get(site_path, KEY_SHAREPOINT)
assert r.status_code == 200, f"기대 200, 실제 {r.status_code}: {r.text[:300]}"
site = r.json()
SITE_ID = site["id"]
print("✅ 사이트 해석 성공")
print("  이름  :", site.get("displayName") or site.get("name"))
print("  웹 URL:", site.get("webUrl"))
print("  siteId:", SITE_ID)

GET /sites/mngenvmcap094463.sharepoint.com:/sites/ContosoIT  → HTTP 200
✅ 사이트 해석 성공
  이름  : Contoso IT
  웹 URL: https://mngenvmcap094463.sharepoint.com/sites/ContosoIT
  siteId: mngenvmcap094463.sharepoint.com,d15ba099-e587-4cfb-a749-f688b6710cda,8965bf4e-7152-4c9b-b661-de1ed9e11961


In [28]:
# 셀 8: [graph-sharepoint 키] GET /sites/{id}/lists → 리스트 목록
r = graph_get(f"/sites/{SITE_ID}/lists", KEY_SHAREPOINT, params={
    "$select": "id,displayName,name,createdDateTime",
})
assert r.status_code == 200, f"기대 200, 실제 {r.status_code}: {r.text[:300]}"
lists = r.json().get("value", [])
print(f"✅ 리스트 {len(lists)}개 조회 성공\n")
print(f"{'#':<3}{'displayName':<30}{'name':<24}created")
print("-" * 80)
for i, l in enumerate(lists, 1):
    print(f"{i:<3}{(l.get('displayName') or '-'):<30}{(l.get('name') or '-'):<24}{l.get('createdDateTime') or '-'}")
LIST_ID = lists[1]["id"] if lists else ""
print("\n다음 항목 조회에 사용할 LIST_ID:", LIST_ID)

GET /sites/mngenvmcap094463.sharepoint.com,d15ba099-e587-4cfb-a749-f688b6710cda,8965bf4e-7152-4c9b-b661-de1ed9e11961/lists  → HTTP 200
✅ 리스트 5개 조회 성공

#  displayName                   name                    created
--------------------------------------------------------------------------------
1  문서                            Shared Documents        2026-02-23T15:10:21Z
2  장치                            List                    2026-02-23T15:10:35Z
3  티켓                            List1                   2026-02-23T15:10:44Z
4  웹 템플릿 확장                      wte                     2026-02-23T15:10:33Z
5  이벤트                           Events                  2026-02-23T15:10:50Z

다음 항목 조회에 사용할 LIST_ID: 5c4ce3e0-c774-4559-be68-3e00eb078ae7


In [29]:
# 셀 9: [graph-sharepoint 키] GET /sites/{id}/lists/{listId}/items?$expand=fields → 항목(행)
assert LIST_ID, "조회할 리스트가 없습니다. 셀 8에서 리스트를 먼저 확인하세요."
r = graph_get(f"/sites/{SITE_ID}/lists/{LIST_ID}/items", KEY_SHAREPOINT, params={
    "$expand": "fields",
    "$top": 5,
})
assert r.status_code == 200, f"기대 200, 실제 {r.status_code}: {r.text[:300]}"
items = r.json().get("value", [])
print(f"✅ 항목 {len(items)}개 조회 성공 (상위 5개)\n")
for i, it in enumerate(items, 1):
    fields = it.get("fields", {})
    title = fields.get("Title") or fields.get("LinkTitle") or "-"
    shown = {k: v for k, v in fields.items() if not k.startswith('@')}
    print(f"[{i}] id={it.get('id')}  Title={title}")
    print("    fields:", shown)

GET /sites/mngenvmcap094463.sharepoint.com,d15ba099-e587-4cfb-a749-f688b6710cda,8965bf4e-7152-4c9b-b661-de1ed9e11961/lists/5c4ce3e0-c774-4559-be68-3e00eb078ae7/items  → HTTP 200
✅ 항목 4개 조회 성공 (상위 5개)

[1] id=1  Title=Surface Laptop 13
    fields: {'Title': 'Surface Laptop 13', 'DevicePhoto': '{"fileName":"Reserved_ImageAttachment_[11]_[DevicePhoto][32]_[b9e36b17ddda4cdcac03104a4f762b11][1]_[8].png","originalImageName":"Surface-Laptop-13"}', 'LinkTitleNoMenu': 'Surface Laptop 13', 'LinkTitle': 'Surface Laptop 13', 'Status': '사용 가능', 'Manufacturer': 'Fabrikam, Inc.', 'Model': 'ABC-1000', 'AssetType': '노트북', 'Color': '검은색', 'SerialNumber': 'BAD0123DDFA2', 'PurchaseDate': '2026-02-27T08:00:00Z', 'PurchasePrice': 150000.0, 'OrderNumber': 'PO00001', 'CurrentOwnerLookupId': '11', 'PreviousOwnerLookupId': '11', 'DueDate': '2026-02-28T08:00:00Z', 'ConditionNotes': '이상 없음', 'Image': {'Description': 'https://raw.githubusercontent.com/microsoft/agent-academy/refs/heads/main/docs/recruit/00-course-

In [30]:
# 셀 10: [graph-users 키]로 SharePoint 접근 시도 → 403 기대 (정책 격리)
r = graph_get("/sites/root", KEY_USERS)
assert r.status_code == 403, f"기대 403, 실제 {r.status_code}: {r.text[:300]}"
print("✅ graph-users 키의 SharePoint 접근이 정책으로 차단됨 (403)")
print("   → 구독 키마다 접근 가능한 데이터가 다름을 확인")

GET /sites/root  → HTTP 403
✅ graph-users 키의 SharePoint 접근이 정책으로 차단됨 (403)
   → 구독 키마다 접근 가능한 데이터가 다름을 확인


## Part 2 (옵션 B): 권한별 UAMI

`scripts/deploy-graph-uami.sh` 실행 + 정책을 Part 2 버전(`client-id` 라우팅)으로 교체한 뒤 실행하세요.

이제 격리는 **정책이 아니라 토큰 권한 자체**로 이뤄집니다.
`graph-users` 경로가 잘못 열려도, 그 UAMI 토큰에는 `Mail.Read`가 없어 **Graph가 직접 거부**합니다.

In [31]:
# 셀 11: [graph-users 키] 메일 접근 재시도 → Graph 권한 거부 기대 (403)
# Part 1과 달리, 이 403은 APIM 정책이 아니라 Graph가 "권한 없음"으로 반환합니다.
# (정책 게이트를 제거/완화한 Part 2 정책에서 테스트)
r = graph_get(f"/users/{TEST_USER_ID}/messages", KEY_USERS)
print("본문 일부:", r.text[:300])
assert r.status_code in (401, 403), f"기대 401/403(Graph 권한 거부), 실제 {r.status_code}"
print("✅ UAMI 토큰에 Mail.Read가 없어 Graph가 거부 — 진짜 격리 확인")

GET /users/cb319021-e447-4dfb-837a-60042cc7a8a9/messages  → HTTP 403
본문 일부: {"error":{"code":"ErrorAccessDenied","message":"Access is denied. Check credentials and try again."}}
✅ UAMI 토큰에 Mail.Read가 없어 Graph가 거부 — 진짜 격리 확인


In [32]:
# 셀 12: [Part 2] 정상 경로 — 각 UAMI가 자기 권한으로 정상 동작
# 옵션 A와 동일하게 세 경로 모두 200. 차이는 "누가 토큰을 발급했나"입니다.
print("=== 옵션 B 정상 경로 (구독별 UAMI 토큰) ===")

r = graph_get("/users", KEY_USERS, params={"$top": 3})
print(f"users 키       → /users            : HTTP {r.status_code} (사용자 {len(r.json().get('value', []))}명)")
assert r.status_code == 200

r = graph_get(f"/users/{TEST_USER_ID}/messages", KEY_MAIL, params={"$top": 3, "$select": "subject"})
print(f"mail 키        → /messages         : HTTP {r.status_code} (메일 {len(r.json().get('value', []))}건)")
assert r.status_code == 200

r = graph_get(f"/sites/{SITE_ID}/lists", KEY_SHAREPOINT)
lists = r.json().get("value", [])
print(f"sharepoint 키  → /sites/{{id}}/lists : HTTP {r.status_code} (리스트 {len(lists)}개)")
for l in lists:
    print("   -", l.get("displayName"))
assert r.status_code == 200 and len(lists) > 0

print("✅ 각 UAMI가 자기 권한 범위에서 정상 동작 (users→User.Read.All, mail→Mail.Read, sharepoint→Sites.Read.All)")

=== 옵션 B 정상 경로 (구독별 UAMI 토큰) ===
GET /users  → HTTP 200
users 키       → /users            : HTTP 200 (사용자 3명)
GET /users/cb319021-e447-4dfb-837a-60042cc7a8a9/messages  → HTTP 200
mail 키        → /messages         : HTTP 200 (메일 3건)
GET /sites/mngenvmcap094463.sharepoint.com,d15ba099-e587-4cfb-a749-f688b6710cda,8965bf4e-7152-4c9b-b661-de1ed9e11961/lists  → HTTP 200
sharepoint 키  → /sites/{id}/lists : HTTP 200 (리스트 5개)
   - 문서
   - 장치
   - 티켓
   - 웹 템플릿 확장
   - 이벤트
✅ 각 UAMI가 자기 권한 범위에서 정상 동작 (users→User.Read.All, mail→Mail.Read, sharepoint→Sites.Read.All)


In [33]:
# 셀 13: [Part 2] 교차 접근 차단 — 정책이 아니라 '토큰 권한 부재'로 막힘
# Part 1의 403은 APIM 정책이 반환했지만, 여기서는 Graph/SharePoint가 직접 거부합니다.
print("=== 옵션 B 교차 접근 차단 (토큰에 해당 권한이 없음) ===")

# 1) users 키로 메일 → Graph가 ErrorAccessDenied 반환
r = graph_get(f"/users/{TEST_USER_ID}/messages", KEY_USERS)
err = r.json().get("error", {}).get("code", "")
print(f"users 키 → /messages        : HTTP {r.status_code}  {err}")
assert r.status_code in (401, 403), f"기대 401/403, 실제 {r.status_code}"

# 2) users 키로 사이트(by id) → 403 accessDenied
r = graph_get(f"/sites/{SITE_ID}", KEY_USERS)
err = r.json().get("error", {}).get("code", "")
print(f"users 키 → /sites/{{id}}      : HTTP {r.status_code}  {err}")
assert r.status_code in (401, 403), f"기대 401/403, 실제 {r.status_code}"

# 3) users 키로 리스트 컬렉션 → 200이지만 데이터 미노출(빈 배열)
#    SharePoint는 권한 없는 앱 토큰에 컬렉션을 200+빈결과로 돌려줍니다.
r = graph_get(f"/sites/{SITE_ID}/lists", KEY_USERS)
empty = r.json().get("value", [])
print(f"users 키 → /sites/{{id}}/lists : HTTP {r.status_code}  (리스트 {len(empty)}개 — 실제 데이터 미노출)")
assert len(empty) == 0, f"격리 실패: 리스트 {len(empty)}개가 노출됨"

print("✅ 정책이 아닌 토큰 권한 자체로 격리 — 잘못된 키로는 데이터가 실제로 나오지 않음")

=== 옵션 B 교차 접근 차단 (토큰에 해당 권한이 없음) ===
GET /users/cb319021-e447-4dfb-837a-60042cc7a8a9/messages  → HTTP 403
users 키 → /messages        : HTTP 403  ErrorAccessDenied
GET /sites/mngenvmcap094463.sharepoint.com,d15ba099-e587-4cfb-a749-f688b6710cda,8965bf4e-7152-4c9b-b661-de1ed9e11961  → HTTP 403
users 키 → /sites/{id}      : HTTP 403  accessDenied
GET /sites/mngenvmcap094463.sharepoint.com,d15ba099-e587-4cfb-a749-f688b6710cda,8965bf4e-7152-4c9b-b661-de1ed9e11961/lists  → HTTP 200
users 키 → /sites/{id}/lists : HTTP 200  (리스트 0개 — 실제 데이터 미노출)
✅ 정책이 아닌 토큰 권한 자체로 격리 — 잘못된 키로는 데이터가 실제로 나오지 않음


### Part 2 증빙: 정책을 바꿨으니 "차단 방식"도 바뀌어야 한다

옵션 A는 **APIM 정책**이 경로를 막았고(한글 메시지), 옵션 B는 그 정책 게이트를 제거했습니다.
따라서 같은 요청이라도 **차단의 출처**가 달라지고, 일부는 **상태코드 자체가 바뀝니다**.

In [34]:
# 셀 14: [Part 2 증빙] 옵션 A와 무엇이 달라졌나 — 같은 403이라도 '출처'가 다르다
# 옵션 A: APIM 정책이 return-response로 403 + 한글 메시지("이 구독은 ...만 허용됩니다")
# 옵션 B: 요청이 Graph까지 도달하고, 토큰에 권한이 없어 Graph/SharePoint가 직접 거부
print("=== 403의 '출처'가 APIM 정책 → 토큰 권한으로 바뀐 증거 ===")
for label, path, key in [
    ("users키 → /messages ", f"/users/{TEST_USER_ID}/messages", KEY_USERS),
    ("mail키  → /users    ", "/users",                          KEY_MAIL),
    ("users키 → /sites/root", "/sites/root",                    KEY_USERS),
]:
    r = graph_get(path, key)
    code = r.json().get("error", {}).get("code", "")
    print(f"   {label} HTTP {r.status_code}  error.code = {code}")
print("→ error.code = ErrorAccessDenied / Authorization_RequestDenied / accessDenied")
print("  = APIM 정책 메시지가 아니라 Graph/SharePoint가 낸 표준 오류 (옵션 A와 본문이 다름)")

print()
print("=== 정책 게이트 제거로 상태코드 자체가 바뀐 케이스 ===")
r = graph_get(f"/sites/{SITE_ID}/lists", KEY_USERS)
n = len(r.json().get("value", []))
print(f"   users키 → /sites/{{id}}/lists")
print(f"     옵션 A: HTTP 403 (APIM 정책이 차단)")
print(f"     옵션 B: HTTP {r.status_code} (리스트 {n}개)  ← 게이트가 없어 Graph까지 도달, 단 권한이 없어 빈 결과")
assert n == 0, f"격리 실패: 리스트 {n}개 노출"
print("→ 상태코드는 403→200으로 바뀌었지만, 토큰에 Sites 권한이 없어 데이터는 여전히 미노출")

=== 403의 '출처'가 APIM 정책 → 토큰 권한으로 바뀐 증거 ===
GET /users/cb319021-e447-4dfb-837a-60042cc7a8a9/messages  → HTTP 403
   users키 → /messages  HTTP 403  error.code = ErrorAccessDenied
GET /users  → HTTP 403
   mail키  → /users     HTTP 403  error.code = Authorization_RequestDenied
GET /sites/root  → HTTP 403
   users키 → /sites/root HTTP 403  error.code = accessDenied
→ error.code = ErrorAccessDenied / Authorization_RequestDenied / accessDenied
  = APIM 정책 메시지가 아니라 Graph/SharePoint가 낸 표준 오류 (옵션 A와 본문이 다름)

=== 정책 게이트 제거로 상태코드 자체가 바뀐 케이스 ===
GET /sites/mngenvmcap094463.sharepoint.com,d15ba099-e587-4cfb-a749-f688b6710cda,8965bf4e-7152-4c9b-b661-de1ed9e11961/lists  → HTTP 200
   users키 → /sites/{id}/lists
     옵션 A: HTTP 403 (APIM 정책이 차단)
     옵션 B: HTTP 200 (리스트 0개)  ← 게이트가 없어 Graph까지 도달, 단 권한이 없어 빈 결과
→ 상태코드는 403→200으로 바뀌었지만, 토큰에 Sites 권한이 없어 데이터는 여전히 미노출


## Part 1 vs Part 2 요약

| 구분 | 옵션 A (System MI + 정책) | 옵션 B (권한별 UAMI) |
|------|--------------------------|----------------------|
| 실제 권한 경계 | MI 1개 = 모든 권한 합집합 | UAMI별 단일 권한 |
| 차단 주체 | APIM 정책 (403) | Graph 자체 (토큰에 권한 없음) |
| 정책 우회 시 | 전체 권한 노출 위험 | 여전히 안전 (토큰에 권한 없음) |
| 구성 복잡도 | 낮음 (MI 1개) | 높음 (UAMI 3개 + attach) |

**공통점:** 클라이언트는 두 경우 모두 구독 키만 보내며, Graph 토큰을 직접 발급하지 않습니다.   
**결론:** 최소 권한/강한 격리가 필요하면 옵션 B, 간단한 데모/내부용이면 옵션 A.